# Loppuprojekti  

## Rekisterinumeron tunnistus OCR-mallilla 

Samuli Lamminmäki, Ismet Ymeri, Stefanos Thomas, Onni Luova, Jan Nässling 

# Yleiskatsaus 

Projekti toteuttaa rekisterikilpien merkkien tunnistusta syötetyistä kuvista. Järjestelmä käyttää räätälöityä OCR-mallia numeroiden ja kirjainten tunnistamiseen rekisterikilvistä. 

## Datasetin läpikäynti 

Projektissa käytetään Licence Plate Text Recognition datasettiä. Datasetin tekijä on Nick David Yazdan. 

Datasetin sisältö:  

    20 000 erilaatuista kuvaa autojen rekisterikilvistä 

    Kuvien laatu vaihtelee huomattavasti, mikä on hyödyllistä mallin kouluttamisessa 

    Datasettiin kuuluu CSV-tiedosto, joka sisältää rekisterinumerot jokaiselle kuvalle 

    Datasetissä on 24 puuttuvaa tunnistetta (labelia) 

Linkki datasettiin: https://www.kaggle.com/datasets/nickyazdani/license-plate-text-recognition-dataset

In [19]:
import kagglehub

path = kagglehub.dataset_download("nickyazdani/license-plate-text-recognition-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\onnil\.cache\kagglehub\datasets\nickyazdani\license-plate-text-recognition-dataset\versions\1


## Datan lataus ja esikäsittely 

Järjestelmä lataa rekisterikilpikuvat ja niiden vastaavat tekstimuotoiset tunnisteet CSV-tiedostosta. Jokaisella kuvalla on nämä esikäsittelyvaiheet: 

    Grayscaling: Kuvat muutetaan mustavalkoisiksi, jotta malli ei keskity liikaa väreihin 

    Kontrastin venytys: Parantaa kuvan kontrastia parempaa piirteiden tunnistusta varten 

    Koon muutos: Kuvat muutetaan kokoon 128×96 pikseliä käyttäen kuutiointerpolaatiota 

    Normalisointi: Pikseliarvot normalisoidaan välille 0-1

In [20]:
import cv2
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from keras.utils import pad_sequences

csv_path = "/home/samul/.cache/kagglehub/datasets/nickyazdani/license-plate-text-recognition-dataset/versions/1/lpr.csv"
img_dir = "/home/samul/.cache/kagglehub/datasets/nickyazdani/license-plate-text-recognition-dataset/versions/1/cropped_lps/cropped_lps"

## Merkkien koodaus 

Rekisterikilven merkit koodataan mallin koulutusta varten: 

    Tuetut merkit: isot kirjaimet A-Z ja numerot 0-9 (yhteensä 36 merkkiä) 

    One-hot-enkoodaus: käytetään mallin koulutukseen 


In [21]:
characters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 -"
char_to_index = {char: idx for idx, char in enumerate(characters)}
index_to_char = {idx: char for char, idx in char_to_index.items()}
num_classes = len(characters) + 1

df = pd.read_csv(csv_path)

def encode_label(text):
    return [char_to_index[c] for c in text.upper() if c in char_to_index]

def contrast_stretching(image):
    min_val = np.min(image)
    max_val = np.max(image)
    if max_val == min_val:
        return image
    stretched_image = (image - min_val) * (255 / (max_val - min_val))
    return stretched_image.astype(np.uint8)

def preprocess_images(img_dir):
    images, labels, label_lengths = [], [], []
    skipped_count = 0
    success_count = 0
    for idx, row in df.iterrows():
        try:
            img_path = os.path.join(img_dir, row["images"])
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                skipped_count += 1
                continue

            img = contrast_stretching(img)
            img = cv2.resize(img, (128, 96), interpolation=cv2.INTER_CUBIC)

            if np.max(img) != np.min(img):
                img = (img - np.min(img)) / (np.max(img) - np.min(img))

            img = (img * 255).astype(np.uint8)
            img = np.expand_dims(img, axis=-1)
            images.append(img)
            encoded = encode_label(row["labels"])
            labels.append(encoded)
            label_lengths.append(len(encoded))
            success_count += 1

        except Exception as e:
            print(f"Virhe käsiteltäessä kuvaa {row['images']}: {str(e)}")

    if not images:
        exit(1)

    print(f"Kuvia käsitelty onnistuneesti: {success_count}")
    return images, labels, label_lengths

images, labels, label_lengths = preprocess_images(img_dir)

max_label_len = max(label_lengths)
padded_labels = pad_sequences(labels, maxlen=max_label_len, value=-1, padding="post")

X = np.array(images)
y = np.array(padded_labels)
label_lengths = np.array(label_lengths)

print(f"Datasetin koko: {X.shape[0]} kuvaa, mitat: {X.shape[1:]} pikseleissä")
print(f"Pisimmän labelin pituus: {max_label_len}")

X_temp, X_test, y_temp, y_test, len_temp, len_test = train_test_split(
    X, y, label_lengths, test_size=0.15, random_state=1234
)

X_train, X_val, y_train, y_val, len_train, len_val = train_test_split(
    X_temp, y_temp, len_temp, test_size=0.1765, random_state=1234
)

time_steps = 96 // 8

train_input_len = np.full((len(X_train), 1), time_steps, dtype=np.int32)
val_input_len = np.full((len(X_val), 1), time_steps, dtype=np.int32)
test_input_len = np.full((len(X_test), 1), time_steps, dtype=np.int32)

FileNotFoundError: [Errno 2] No such file or directory: '/home/samul/.cache/kagglehub/datasets/nickyazdani/license-plate-text-recognition-dataset/versions/1/lpr.csv'

In [ ]:
print(X.shape)
print(X_train.shape)
print(X_val.shape)

## Mallin arkkitehtuuri 

OCR-malli on hybridi CNN-LSTM-arkkitehtuuri: 

    Piirteiden erottelu:  

    Useita CNN-kerroksia batch-normalisaatiolla ja max poolingilla 

    Erottaa visuaaliset piirteet rekisterikilpien kuvista 

 

    Sekvenssin käsittely:  

    Kaksisuuntaiset LSTM-kerrokset käsittelevät CNN-piirteitä sekvensseinä 

    Tallentaa kontekstuaalisen tiedon merkkien välillä 

 

Lähtökerros: 

    Dense-kerros softmax-aktivaatiolla merkkien luokitteluun 

    Yksi lähtö per aikavaihe, joka edustaa merkkien todennäköisyyksiä 

In [ ]:
from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, Reshape, Dense, Dropout, BatchNormalization, Activation
from keras.layers import Bidirectional, LSTM
from keras.layers import Lambda
from keras.layers import RandomFlip, RandomRotation, RandomZoom
from keras._tf_keras.keras.callbacks import EarlyStopping
import os
import tensorflow as tf

os.environ["KERAS_BACKEND"] = "tensorflow"

# Määritellään tärkeät parametrit
img_height, img_width, img_channels = 96, 128, 1
num_classes = 36 + 1  # 26 kirjainta + 10 numeroa + CTC-blank

# Syötekuva
input_img = Input(shape=(img_height, img_width, img_channels), name='image_input')
x = RandomFlip("horizontal")(input_img)
x = RandomRotation(0.05)(x)
x = RandomZoom(0.1)(x)

## CNN Feature extractor
x = Conv2D(64, (5, 5), padding='same', kernel_initializer='he_normal')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)  # -> (48, 64)

x = Conv2D(128, (3, 3), padding='same', kernel_initializer='he_normal')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)  # -> (24, 32)

x = Conv2D(256, (3, 3), padding='same', kernel_initializer='he_normal')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
x = MaxPooling2D(pool_size=(2, 2))(x)  # -> (12, 16)

x = Conv2D(512, (3, 3), padding='same', kernel_initializer='he_normal')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)

x = Conv2D(512, (3, 3), padding='same', kernel_initializer='he_normal')(x)
x = BatchNormalization()(x)
x = Activation('relu')(x)
# Ei poolingia tässä — säilytetään enemmän "aikaa"

# Muodosta sekvenssi aikadimensiossa (width)
# Muoto: (batch, height, width, channels) → (batch, width, height × channels)
new_shape = (x.shape[2], x.shape[1] * x.shape[3])  # (W, H*C)
x = Reshape(target_shape=new_shape)(x)

# 2x Bidirectional LSTM
x = Bidirectional(LSTM(256, return_sequences=True))(x)
x = Dropout(0.25)(x)
x = Bidirectional(LSTM(256, return_sequences=True))(x)

# Jokaiselle aikastepille merkki (softmax yli kirjainten ja numeroiden)
x = Dense(num_classes, activation='softmax')(x)

# Malli
model = Model(inputs=input_img, outputs=x)
model.summary()

## Datan Valmistelu 

    Kuvat ja tunnisteet ladataan datasetistä

    Dataset jaetaan koulutus- (80%) ja validointisetteihin (20%) 

    Tunnisteet täydennetään yhtenäiseen pituuteen eräkäsittelyä varten 
    
    CTC (Connectionist Temporal Classification) -menetelmää käytetään vaihtuvanpituisten sekvenssien käsittelyyn

In [ ]:
from tensorflow.keras.optimizers import Adam
from keras.callbacks import ReduceLROnPlateau

# Placeholderit CTC:lle
labels = Input(name='label', shape=(None,), dtype='int32')
input_length = Input(name='input_length', shape=(1,), dtype='int32')  # output sequence length
label_length = Input(name='label_length', shape=(1,), dtype='int32')  # always 6

# CTC-loss
def ctc_lambda_func(args):
    y_pred, labels, input_length, label_length = args
    return tf.keras.backend.ctc_batch_cost(labels, y_pred, input_length, label_length)

# Output of the Lambda layer
loss_out = Lambda(ctc_lambda_func, output_shape=(1,), name='ctc_loss')(
    [x, labels, input_length, label_length]
)

# Lopullinen training-malli
ctc_model = Model(inputs=[input_img, labels, input_length, label_length], outputs=loss_out)

optimizer = Adam(learning_rate=0.0001)  # Smaller, more stable learning rate

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

ctc_model.compile(
    optimizer=optimizer,
    loss=lambda y_true, y_pred: y_pred
)

# Muotoile data
train_input_len = train_input_len.astype(np.int32)
val_input_len = val_input_len.astype(np.int32)

train_data = {
    'image_input': X_train,
    'label': y_train,
    'input_length': train_input_len,
    'label_length': np.expand_dims(len_train, axis=1)
}
val_data = {
    'image_input': X_val,
    'label': y_val,
    'input_length': val_input_len,
    'label_length': np.expand_dims(len_val, axis=1)
}

## Koulutuksen asetukset 

    Häviöfunktio: CTC-häviö 

    Optimointi: Adam

    Aikainen pysäytys: Validointihäviön seuranta, kärsivällisyys 10 epookkia 
    
    Eräkoko: 100 näytettä 

    Epookit: 150 (riippuen aikaisesta pysäytyksestä) 

In [ ]:
# Dummy-arvot ilman sanakirjaa
train_dummy_y = np.zeros((len(X_train), 1), dtype=np.float32)
val_dummy_y = np.zeros((len(X_val), 1), dtype=np.float32)

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Suorita koulutus
history = ctc_model.fit(
    x=train_data,
    y=train_dummy_y,
    validation_data=(val_data, val_dummy_y),
    epochs=150,
    batch_size=100,
    callbacks=[early_stopping],
    verbose=0
)

model.save("trained_license_plate_model.h5")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))  # Voit säätää korkeutta halutessasi
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

characters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 -"

# Dekoodausfunktio sekvensseille
def decode_sequence(seq):
    return ''.join([characters[i] for i in seq if i != -1 and i < len(characters)])

# Tarkistetaan mallin ulostulo
print(model.output_shape)

# Ennusteet mallilta
preds = model.predict(X_test)
decoded, _ = tf.keras.backend.ctc_decode(
    preds,
    input_length=np.ones(preds.shape[0]) * preds.shape[1],
    greedy=False,
    beam_width=10)

decoded_sequences = decoded[0].numpy()

correct_chars = 0
total_chars = 0

# Näytä ensimmäiset 10 kuvaa
for i in range(10):
    img = X_test[i].squeeze()
    prediction = decode_sequence(decoded_sequences[i])
    true_label = decode_sequence(y_test[i])

    # Merkkikohtainen vertailu
    for pred_char, true_char in zip(prediction, true_label):
        if pred_char == true_char:
            correct_chars += 1
    total_chars += len(true_label)

    print(f"Image {i + 1}:")
    print(f"  Prediction   : {prediction}")
    print(f"  True value   : {true_label}")

    plt.imshow(img, cmap='gray')
    plt.title(f"P: {prediction} | T: {true_label}")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Loput testidatasta
for i in range(10, len(X_test)):
    pred = decode_sequence(decoded_sequences[i])
    true = decode_sequence(y_test[i])
    for pred_char, true_char in zip(pred, true):
        if pred_char == true_char:
            correct_chars += 1
    total_chars += len(true)

accuracy = correct_chars / total_chars
print(f"\nTest set **character-level** accuracy: {accuracy:.2%}")

# Mallin evaluointi 

Mallin testitarkkuus on noin 73%. Malli toimii hyvin jopa vähän huonolaatuisempiin kuviin. Datasetin laatu on keskinkertainen, joka vaikuttaa mallin koulutuksen laatuun. 

In [ ]:
img = "37465.jpg"

def decode_single_sequence(seq):
    return ''.join([characters[int(i)] for i in seq if i != -1 and i < len(characters)])

def preprocess_single_image(image_path):
    try:
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise ValueError("Kuvaa ei voitu lukea.")

        min_val, max_val = np.min(img), np.max(img)
        if max_val != min_val:
            img = (img - min_val) * (255 / (max_val - min_val))
            img = img.astype(np.uint8)

        img = cv2.resize(img, (128, 96), interpolation=cv2.INTER_CUBIC)

        if np.max(img) != np.min(img):
            img = (img - np.min(img)) / (np.max(img) - np.min(img))

        img = (img * 255).astype(np.uint8)
        img = np.expand_dims(img, axis=-1)  # (96, 128, 1)

        return img

    except Exception as e:
        print(f"Virhe kuvan käsittelyssä ({image_path}): {e}")
        return None


processed_img = preprocess_single_image(img)
input_tensor = np.expand_dims(processed_img, axis=0)
pred = model.predict(input_tensor)

decoded_image, _ = tf.keras.backend.ctc_decode(
    pred,
    input_length=np.ones(pred.shape[0]) * pred.shape[1],
    greedy=False,
    beam_width=10)

decoded_sequence = decoded_image[0].numpy()
image = processed_img.squeeze()
prediction = decode_single_sequence(decoded_sequence[0].flatten())
print(f"Image:")
print(f"  Prediction   : {prediction}")

plt.imshow(image, cmap='gray')
plt.title(f"P: {prediction}")
plt.axis('off')
plt.tight_layout()
plt.show()